# Feature generation with an Autoencoder
The goal of this notebook is to generate features from the spectrograms of different mic channels.  
For this, an autoencoder will be used to encode the spectrograms into a low dimensional vector,  
which will later be passed to the classification model.

To produce enough training samples, spectrograms are generated from 2-second sliding windows across the recordings.

For this notebook we utilize a simpele train-test split where we leave out a car to validate the model.
For the final classifier the training of the autoencoder should adhere to the choosen split.

## Conditioning the latent vectors to be discriminative
To bias the latent representations to be better suited for later classification, we utilize a classifier that contributes to the loss.  
$\alpha$ controls the impact of the classifier.

In [1]:
# TODOS
# Instead of using the mel spectrogram, use the meal coefficients directly as features
# Apply some normalization
# Use 1D convolutions per mfcc feature over time
# Use Upconv in reconstruction

In [2]:
from pathlib import Path

import librosa
import lightning as pl
import numpy as np
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from tqdm import tqdm

from roar import ROOT_DIR
from roar.preprocessing import load_h5_channel

# Data Module
To keep all dataloading in one place, we will use a pytorch lighning datamodule

# Multi-channel Implementation
Multiple channels MFCCs are now stacked in the input with shape [num_channels, n_mfcc, time_frames]


In [ ]:
def generate_spectrograms(file: Path, channel_names: list[str]) -> list[Tensor]:
    """Generate multi-channel spectrograms from audio files.

    This function processes each channel at its native sample rate, computes MFCCs,
    then aligns them by interpolating to a common number of frames. This is more
    efficient than resampling raw audio and preserves feature quality.

    Args:
        file: Path to H5 file
        channel_names: List of channel names to process

    Returns:
        List of tensors with shape [num_channels, n_mfcc, time_frames]
        All tensors will have consistent time_frames dimension (188 frames)
    """
    window_duration = 2.0
    # step_size = 0.1
    n_mfcc = 13

    # Fixed window size in MFCC frames for consistency across all files
    # With hop_length=512 and typical sample rates ~48-51kHz:
    # 2 seconds ≈ 188 frames (avg: 48kHz/512 * 2s ≈ 187.5)
    WINDOW_FRAMES = 188
    # Step size: 0.1s ≈ 9-10 frames, use 10 for consistency
    STEP_FRAMES = 10

    # Process each channel independently at native sample rate
    channel_mfcc_full = []  # Full MFCCs for entire recording
    channel_info = []  # Store metadata for each channel

    for channel_name in channel_names:
        try:
            data, sample_rate = load_h5_channel(file, channel_name)  # type: ignore
            data = data.astype(np.float32)
            sample_rate = int(round(sample_rate))

            # Check if data is long enough for at least one window
            window_samples = int(window_duration * sample_rate)
            if len(data) < window_samples:
                channel_info.append(None)
                channel_mfcc_full.append(None)
                continue

            # Apply preprocessing
            sos = scipy.signal.butter(4, 200, "hp", fs=sample_rate, output="sos")
            data = scipy.signal.sosfilt(sos, data)
            data = data / (np.max(np.abs(data)) + 1e-8)  # Normalize

            # Compute MFCC for full recording at native sample rate
            mfcc = librosa.feature.mfcc(
                y=data, sr=sample_rate, n_mfcc=n_mfcc, n_fft=2048, hop_length=512
            )

            channel_mfcc_full.append(mfcc)
            channel_info.append(
                {
                    "sample_rate": sample_rate,
                    "duration": len(data) / sample_rate,
                    "n_frames": mfcc.shape[1],
                }
            )

        except KeyError:
            # Channel missing
            channel_mfcc_full.append(None)
            channel_info.append(None)

    # Check if at least one channel loaded successfully
    valid_mfccs = [m for m in channel_mfcc_full if m is not None]
    if not valid_mfccs:
        print(f"No valid channels found in file: {file}")
        return []

    # Determine target number of frames (use minimum to avoid extrapolation)
    target_frames = min(m.shape[1] for m in valid_mfccs)

    # Check if we have enough frames for at least one window
    if target_frames < WINDOW_FRAMES:
        valid_info = [info for info in channel_info if info is not None]
        avg_sample_rate = np.mean([info["sample_rate"] for info in valid_info])
        frames_per_second = avg_sample_rate / 512
        duration_sec = target_frames / frames_per_second
        print(
            f"Audio too short ({duration_sec:.2f}s, {target_frames} frames < {WINDOW_FRAMES} frames), discarding: {file}"
        )
        return []

    # Align all MFCCs to target frame count
    aligned_mfccs = []
    for i, mfcc in enumerate(channel_mfcc_full):
        if mfcc is None:
            # Create zero-filled MFCC for missing channel
            aligned_mfccs.append(np.zeros((n_mfcc, target_frames), dtype=np.float32))
        elif mfcc.shape[1] != target_frames:
            # Interpolate MFCC to target frame count
            zoom_factor = target_frames / mfcc.shape[1]
            mfcc_resampled = scipy.ndimage.zoom(mfcc, (1, zoom_factor), order=1)
            # Ensure exact size (zoom can have rounding issues)
            if mfcc_resampled.shape[1] != target_frames:
                mfcc_resampled = mfcc_resampled[:, :target_frames]
            aligned_mfccs.append(mfcc_resampled)
        else:
            aligned_mfccs.append(mfcc)

    # Generate sliding window spectrograms with fixed window size
    spectrograms = []
    for start_frame in range(0, target_frames - WINDOW_FRAMES + 1, STEP_FRAMES):
        window_mfccs = []
        for mfcc in aligned_mfccs:
            window_mfcc = mfcc[:, start_frame : start_frame + WINDOW_FRAMES]
            window_mfccs.append(window_mfcc)

        # Stack along channel dimension: [num_channels, n_mfcc, WINDOW_FRAMES]
        multi_channel_mfcc = np.stack(window_mfccs, axis=0)
        spectrograms.append(multi_channel_mfcc)

    tensor_spectrograms = [torch.tensor(spec).float() for spec in spectrograms]
    return tensor_spectrograms

In [39]:
class SpectrogramDataset(torch.utils.data.Dataset):
    """
    Dataset for loading spectrogram tensors with track labels.

    Folder structure:
        data_images → trackXXX → CarYYY → TyreZZZ → experiment_AAA → spec_000.pt

    Target: Track name (trackXXX)
    """

    def __init__(
        self,
        tensor_paths: list[Path],
        mean: torch.Tensor,
        std: torch.Tensor,
    ):
        """
        Args:
            tensor_paths: List of paths to .pt files
            mean: Mean tensor for normalization, shape [num_channels, n_mfcc, 1]
            std: Std tensor for normalization, shape [num_channels, n_mfcc, 1]
        """
        self.tensor_paths = tensor_paths
        self.mean = mean
        self.std = std

        # Extract track labels and create label mapping
        self.track_labels = self._extract_track_labels()
        self.unique_tracks = sorted(set(self.track_labels))
        self.track_to_idx = {track: idx for idx, track in enumerate(self.unique_tracks)}
        self.idx_to_track = {idx: track for track, idx in self.track_to_idx.items()}

        print(f"Dataset initialized with {len(self.tensor_paths)} samples")
        print(f"Number of unique tracks: {len(self.unique_tracks)}")
        print(f"Tracks: {self.unique_tracks}")
        print(f"Class distribution: {self._get_class_distribution()}")

    def _extract_track_labels(self) -> list[str]:
        """Extract track name from each file path."""
        labels = []
        for path in self.tensor_paths:
            # Path structure: .../trackXXX/CarYYY/TyreZZZ/experiment_AAA/spec_000.pt
            parts = path.parts
            # Find the track part (should be 5 levels up from the file)
            track_idx = (
                -5
            )  # spec_000.pt (-1), experiment_AAA (-2), TyreZZZ (-3), CarYYY (-4), trackXXX (-5)
            track = parts[track_idx]
            labels.append(track)
        return sorted(labels)

    def _get_class_distribution(self) -> dict[str, int]:
        """Get the distribution of samples per track."""
        distribution = {}
        for label in self.track_labels:
            distribution[label] = distribution.get(label, 0) + 1
        return distribution

    def normalize_spectrogram(self, spec: torch.Tensor) -> torch.Tensor:
        """
        Normalize spectrogram based on statistics type.

        Args:
            spec: Spectrogram tensor of shape [num_channels, n_mfcc, time_frames]

        Returns:
            Normalized spectrogram of shape [num_channels, n_mfcc, time_frames]
        """
        normalized = (spec - self.mean) / (self.std + 1e-8)
        return normalized

    def __len__(self) -> int:
        """Return the total number of samples."""
        return len(self.tensor_paths)

    def __getitem__(self, idx: int) -> tuple[Tensor, int]:
        """
        Get a single sample.

        Args:
            idx: Index of the sample

        Returns:
            Tuple of (normalized_spectrogram, track_label_idx)
            where normalized_spectrogram has shape [num_channels, n_mfcc, time_frames]
        """
        # Load spectrogram
        spec_path = self.tensor_paths[idx]
        spec = torch.load(spec_path)

        # Normalize
        spec = self.normalize_spectrogram(spec)

        # Get label
        track_name = self.track_labels[idx]
        label = self.track_to_idx[track_name]

        return spec, label

    def get_class_weights(self) -> torch.Tensor:
        """
        Compute class weights for imbalanced datasets.
        Useful for weighted loss functions.

        Returns:
            Tensor of weights for each class
        """
        distribution = self._get_class_distribution()
        total_samples = len(self)
        num_classes = len(self.unique_tracks)

        weights = []
        for track in self.unique_tracks:
            count = distribution[track]
            weight = total_samples / (num_classes * count)
            weights.append(weight)

        return torch.tensor(weights, dtype=torch.float32)

In [40]:
from roar import MIC_CHANNELS


class SpectralDataModule(pl.LightningDataModule):
    def __init__(
        self,
        source_path: Path,
        dest_path: Path = ROOT_DIR / "data_images",
        leave_out_car: str = "02_Q8 e-tron",
        batch_size: int = 32,
    ) -> None:
        super().__init__()
        self.source_path = source_path
        self.dest_path = dest_path
        self.leave_out_car = leave_out_car
        self.batch_size = batch_size
        # Use multiple channels for multi-channel spectrogram generation
        self.channel_names = MIC_CHANNELS

    def prepare_data(self) -> None:
        """
        Generate the spectrograms for each audio file and save them to disk.
        """
        h5_files = list(self.source_path.glob("**/*.h5"))
        target_folders = []

        for h5_file in h5_files:
            parts = h5_file.relative_to(self.source_path).parts
            track = parts[0]  # trackXXX
            car = parts[1]  # CarYYY
            tyre = parts[2]  # TyreZZZ
            experiment = h5_file.stem  # experiment_AAA (without .h5)

            # Create destination folder
            output_dir = self.dest_path / track / car / tyre / experiment
            target_folders.append(output_dir)

        if not all(folder.exists() for folder in target_folders):
            for h5_file, output_dir in tqdm(zip(h5_files, target_folders)):
                if output_dir.exists():
                    continue  # Skip already processed files
                output_dir.mkdir(parents=True, exist_ok=True)
                spectrograms = generate_spectrograms(h5_file, channel_names=self.channel_names)
                # Save spectrograms to img_path or process as needed
                for i, spectrogram in enumerate(spectrograms):
                    save_file = output_dir / f"spec_{i:03d}.pt"
                    torch.save(spectrogram, save_file)

        # Calculate mean and std per channel and per MFCC coefficient
        all_tensors = []
        spectrogram_tensors = list(self.dest_path.glob("**/spec_*.pt"))
        train_tensors = [t for t in spectrogram_tensors if self.leave_out_car not in str(t)]
        for train_tensor in train_tensors:
            spectrogram = torch.load(train_tensor)
            all_tensors.append(spectrogram)

        # Stack along time dimension: [num_samples, num_channels, n_mfcc, time_frames]
        full_data = torch.stack(all_tensors, dim=0)

        # Compute mean and std per channel and per MFCC coefficient
        # Resulting shape: [num_channels, n_mfcc, 1]
        self.mean = full_data.mean(dim=(0, 3), keepdim=False).unsqueeze(-1)
        self.std = full_data.std(dim=(0, 3), keepdim=False).unsqueeze(-1)

    def setup(self, stage: str | None = None) -> None:
        # For the fit stage, calc the mean and std of the dataset for normalization
        spectrogram_tensors = list(self.dest_path.glob("**/spec_*.pt"))
        if stage == "fit" or stage is None:
            train_tensors = [t for t in spectrogram_tensors if self.leave_out_car not in str(t)]
            val_tensors = [t for t in spectrogram_tensors if self.leave_out_car in str(t)]
            print(
                f"Found {len(train_tensors)} training spectrograms and {len(val_tensors)} validation spectrograms."
            )

            self.train_dataset = SpectrogramDataset(
                tensor_paths=train_tensors,
                mean=self.mean,
                std=self.std,
            )
            self.val_dataset = SpectrogramDataset(
                tensor_paths=val_tensors,
                mean=self.mean,
                std=self.std,
            )
        if stage == "validate":
            val_tensors = [t for t in spectrogram_tensors if self.leave_out_car in str(t)]
            print(f"Found {len(val_tensors)} validation spectrograms.")
            self.val_dataset = SpectrogramDataset(
                tensor_paths=val_tensors,
                mean=self.mean,
                std=self.std,
            )

    def train_dataloader(self) -> torch.utils.data.DataLoader:
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
        )

    def val_dataloader(self) -> torch.utils.data.DataLoader:
        return torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
        )

In [41]:
datamodule = SpectralDataModule(
    source_path=ROOT_DIR / "data_cleaned",
    dest_path=ROOT_DIR / "data_images",
    leave_out_car="02_Q8 e-tron",
    batch_size=8,
)

In [43]:
datamodule.prepare_data()

5it [00:00, 11.26it/s]

Audio too short (1.55s, 138 frames < 188 frames), discarding: /Users/moritzfeik/Developer/ROAR/data_cleaned/track150/02_Q8 e-tron/tyre12/track150_Q8 e-tron_tyre12_meas2_2p5_1_2025-08-15_13-15-55.h5
Audio too short (1.79s, 160 frames < 188 frames), discarding: /Users/moritzfeik/Developer/ROAR/data_cleaned/track150/02_Q8 e-tron/tyre12/track150_Q8 e-tron_tyre12_meas2_2p5_1_2025-08-15_13-15-11.h5


9it [00:00, 11.41it/s]

Audio too short (1.61s, 144 frames < 188 frames), discarding: /Users/moritzfeik/Developer/ROAR/data_cleaned/track150/02_Q8 e-tron/tyre12/track150_Q8 e-tron_tyre12_meas2_2p5_1_2025-08-15_13-14-26.h5


31it [00:03, 12.10it/s]

...

188it [00:21, 10.74it/s]

Audio too short (1.62s, 145 frames < 188 frames), discarding: /Users/moritzfeik/Developer/ROAR/data_cleaned/track211/01_ID.4/tyre1/track211_ID.4_tyre1_meas2_2p5_1_2025-08-07_10-59-40.h5


197it [00:22,  8.90it/s]

Audio too short (1.72s, 154 frames < 188 frames), discarding: /Users/moritzfeik/Developer/ROAR/data_cleaned/track211/04_E-Golf/tyre13/track211_E-Golf_tyre13_meas3_2p5_1_2025-09-26_14-55-08.h5


204it [00:23,  8.61it/s]


## Autoencoder

In [73]:
class ConvAutoencoderClassifier(pl.LightningModule):
    def __init__(
        self,
        latent_dim: int,
        num_classes: int,
        weight: Tensor,
        num_channels: int = 4,
        n_mfcc: int = 13,
        seq_len: int = 188,
        alpha: float = 1.0,
        lr: float = 1e-3,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.alpha = alpha
        self.lr = lr
        self.dropout = dropout
        self.num_channels = num_channels
        self.n_mfcc = n_mfcc
        self.seq_len = seq_len
        self.classifier_criterion = nn.CrossEntropyLoss(weight=weight)
        self.reconstruction_criterion = nn.MSELoss()

        # Encoder: 1D convolutions over time dimension with dropout
        # Input: [B, num_channels * n_mfcc, seq_len] -> Output: [B, 128, seq_len//4]
        input_features = num_channels * n_mfcc
        self.encoder = nn.Sequential(
            nn.Conv1d(input_features, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(2),  # seq_len -> seq_len//2
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(2),  # seq_len//2 -> seq_len//4
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(dropout),
        )

        # Global pooling and latent space projection
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc_latent = nn.Linear(128, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 128)

        # Decoder: 1D transposed convolutions
        # Input: [B, 128, 1] -> Output: [B, num_channels * n_mfcc, seq_len]
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=2, stride=2),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Conv1d(32, input_features, kernel_size=3, padding=1),
        )

        # Classifier head with dropout
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(latent_dim, 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(4, num_classes),
        )

    def encode(self, x: Tensor) -> Tensor:
        """Encode MFCC input to latent representation.

        Args:
            x: Input tensor of shape [B, num_channels, n_mfcc, seq_len]

        Returns:
            Latent representation of shape [B, latent_dim]
        """
        # Flatten channel and MFCC dimensions: [B, num_channels, n_mfcc, seq_len] -> [B, num_channels*n_mfcc, seq_len]
        B, C, M, T = x.shape
        x = x.view(B, C * M, T)

        x = self.encoder(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        z = self.fc_latent(x)
        return z

    def decode(self, z: Tensor, seq_len: int) -> Tensor:
        """Decode latent representation to MFCC reconstruction.

        Args:
            z: Latent tensor of shape [B, latent_dim]
            seq_len: Target sequence length (should be 188)

        Returns:
            Reconstructed MFCC of shape [B, num_channels, n_mfcc, seq_len]
        """
        x = self.fc_decode(z)
        x = x.view(-1, 128, 1)

        # Calculate the size after decoder's transposed convolutions
        # Two stride-2 transposed convs will multiply length by 4
        decoder_output_len = seq_len // 4

        # Interpolate to the size that will result in correct output after decoder
        x = F.interpolate(x, size=decoder_output_len, mode="nearest")

        # Apply decoder
        x = self.decoder(x)

        # Final interpolation to ensure exact match (handles any rounding issues)
        if x.shape[-1] != seq_len:
            x = F.interpolate(x, size=seq_len, mode="linear", align_corners=False)

        # Reshape back to multi-channel format: [B, num_channels*n_mfcc, seq_len] -> [B, num_channels, n_mfcc, seq_len]
        B = x.shape[0]
        x = x.view(B, self.num_channels, self.n_mfcc, seq_len)

        return x

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        """Forward function of the autoencoder

        Args:
            x (Tensor): Input tensor representing MFCC features.
                       Assumed shape is (batch_size, num_channels, n_mfcc, seq_len).

        Returns:
            tuple[Tensor, Tensor, Tensor]: Reconstructed output, classification logits,
                                          and latent representation.
        """
        seq_len = x.shape[-1]
        z = self.encode(x)
        recon = self.decode(z, seq_len)
        logits = self.classifier(z)
        return recon, logits, z

    def compute_loss(
        self,
        x: Tensor,
        recon: Tensor,
        logits: Tensor,
        y: Tensor,
    ) -> tuple[Tensor, Tensor, Tensor]:
        """Compute total loss combining reconstruction MSE and classification cross-entropy.

        Args:
            x (Tensor): Input MFCC features of shape (batch_size, num_channels, n_mfcc, seq_len).
            recon (Tensor): Reconstructed MFCC features matching the shape of `x`.
            logits (Tensor): Classifier logits of shape (batch_size, num_classes).
            y (Tensor): Ground-truth class labels of shape (batch_size,).

        Returns:
            tuple[Tensor, Tensor, Tensor]: Total loss, reconstruction loss, and classification loss.
        """
        recon_loss = self.reconstruction_criterion(recon, x)
        clf_loss = self.classifier_criterion(logits, y)
        total_loss = recon_loss + self.alpha * clf_loss
        return total_loss, recon_loss, clf_loss

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> Tensor:
        x, y = batch
        recon, logits, _ = self.forward(x)
        total_loss, recon_loss, clf_loss = self.compute_loss(x, recon, logits, y)

        self.log("train/total_loss", total_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train/recon_loss", recon_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train/clf_loss", clf_loss, on_step=False, on_epoch=True, prog_bar=True)

        return total_loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> None:
        x, y = batch
        recon, logits, _ = self.forward(x)
        total_loss, recon_loss, clf_loss = self.compute_loss(x, recon, logits, y)

        self.log("val/total_loss", total_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/recon_loss", recon_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/clf_loss", clf_loss, on_step=False, on_epoch=True, prog_bar=True)

    def predict_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int, dataloader_idx: int = 0
    ) -> tuple[Tensor, Tensor]:
        x, _ = batch
        _, logits, z = self.forward(x)
        return torch.argmax(logits, dim=1), z

    def configure_optimizers(self):  # type: ignore
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.3, patience=7
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": lr_scheduler,
                "monitor": "val/recon_loss",
                "step": "epoch",
            },
        }

## Training and Logging

In [65]:
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

In [66]:
datamodule.setup(stage="fit")

Found 2128 training spectrograms and 1110 validation spectrograms.
Dataset initialized with 2128 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 564, 'track211': 1564}
Dataset initialized with 1110 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 153, 'track211': 957}


In [67]:
dl = datamodule.train_dataloader()
for batch in dl:
    break

In [80]:
model = ConvAutoencoderClassifier(
    latent_dim=64,
    num_classes=2,
    num_channels=7,  # Number of channels used
    alpha=0.01,
    lr=3e-4,
    dropout=0.2,  # Added dropout for regularization
    weight=datamodule.train_dataset.get_class_weights(),
)

In [81]:
logger = TensorBoardLogger(ROOT_DIR / "tb_logs", name="autoencoder_classifier")
callbacks = [
    EarlyStopping(monitor="val/recon_loss", patience=20, mode="min"),
    LearningRateMonitor(logging_interval="epoch"),
]
trainer = pl.Trainer(logger=logger, max_epochs=100, callbacks=callbacks)  # type: ignore
model.compile()
trainer.fit(model, datamodule=datamodule)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name                     | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------------------
0 | classifier_criterion     | CrossEntropyLoss  | 0      | train | 0    
1 | reconstruction_criterion | MSELoss           | 0      | train | 0    
2 | encoder                  | Sequential        | 40.1 K | train | 0    
3 | global_pool              | AdaptiveAvgPool1d | 0      | train | 0    
4 | fc_latent                | Linear            | 8.3 K  | train | 0    
5 | fc_decode                | Linear            | 8.3 K  | train | 0    
6 | decoder                  | Sequential        | 29.6 K | train | 0    
7 | classifier               | Sequential  

Found 2128 training spectrograms and 1110 validation spectrograms.
Dataset initialized with 2128 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 564, 'track211': 1564}
Dataset initialized with 1110 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 153, 'track211': 957}
Epoch 99: 100%|██████████| 266/266 [00:06<00:00, 42.22it/s, v_num=13, val/total_loss=0.820, val/recon_loss=0.806, val/clf_loss=1.420, train/total_loss=0.474, train/recon_loss=0.473, train/clf_loss=0.055] 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 266/266 [00:06<00:00, 41.96it/s, v_num=13, val/total_loss=0.820, val/recon_loss=0.806, val/clf_loss=1.420, train/total_loss=0.474, train/recon_loss=0.473, train/clf_loss=0.055]


## Leave one Group Out

In [52]:
from roar import ALL_VEHICLES
import pandas as pd

In [ ]:
# Leave-One-Group-Out Cross Validation
# Train on all vehicles except one, validate on the left-out vehicle

results = []

for left_out_vehicle in ALL_VEHICLES:
    print(f"\n{'=' * 80}")
    print(f"Leave-One-Out: {left_out_vehicle}")
    print(f"{'=' * 80}\n")

    # Create data module with left-out vehicle
    cv_datamodule = SpectralDataModule(
        source_path=ROOT_DIR / "data_cleaned",
        dest_path=ROOT_DIR / "data_images",
        leave_out_car=left_out_vehicle,
        batch_size=8,
    )

    # Prepare and setup data (uses cached spectrograms if already generated)
    cv_datamodule.prepare_data()
    cv_datamodule.setup(stage="fit")

    # Create model with dropout for regularization
    cv_model = ConvAutoencoderClassifier(
        latent_dim=128,
        num_classes=2,
        num_channels=7,
        alpha=0.1,
        lr=1e-3,
        dropout=0.2,  # Added dropout to reduce overfitting
        weight=cv_datamodule.train_dataset.get_class_weights(),
    )

    # Create logger and callbacks
    cv_logger = TensorBoardLogger(
        ROOT_DIR / "tb_logs", name=f"logo_cv_{left_out_vehicle.replace(' ', '_')}"
    )
    cv_callbacks = [
        EarlyStopping(monitor="val/recon_loss", patience=15, mode="min"),
        LearningRateMonitor(logging_interval="epoch"),
    ]

    # Create trainer
    cv_trainer = pl.Trainer(
        logger=cv_logger,
        max_epochs=100,
        callbacks=cv_callbacks,  # type: ignore
        enable_progress_bar=True,
    )

    # Train model
    cv_model.compile()
    cv_trainer.fit(cv_model, datamodule=cv_datamodule)

    # Get training and validation results
    metrics = cv_trainer.callback_metrics

    # Store results
    results.append(
        {
            "left_out_vehicle": left_out_vehicle,
            "train_total_loss": metrics.get("train/total_loss", torch.tensor(float("nan"))).item(),
            "train_recon_loss": metrics.get("train/recon_loss", torch.tensor(float("nan"))).item(),
            "train_clf_loss": metrics.get("train/clf_loss", torch.tensor(float("nan"))).item(),
            "val_total_loss": metrics.get("val/total_loss", torch.tensor(float("nan"))).item(),
            "val_recon_loss": metrics.get("val/recon_loss", torch.tensor(float("nan"))).item(),
            "val_clf_loss": metrics.get("val/clf_loss", torch.tensor(float("nan"))).item(),
            "train_size": len(cv_datamodule.train_dataset),
            "val_size": len(cv_datamodule.val_dataset),
        }
    )

    print(f"\nResults for {left_out_vehicle}:")
    print(f"  Train Total Loss: {results[-1]['train_total_loss']:.4f}")
    print(f"  Train Recon Loss: {results[-1]['train_recon_loss']:.4f}")
    print(f"  Train Clf Loss: {results[-1]['train_clf_loss']:.4f}")
    print(f"  Val Total Loss: {results[-1]['val_total_loss']:.4f}")
    print(f"  Val Recon Loss: {results[-1]['val_recon_loss']:.4f}")
    print(f"  Val Clf Loss: {results[-1]['val_clf_loss']:.4f}")
    print(f"  Train samples: {results[-1]['train_size']}, Val samples: {results[-1]['val_size']}")

# Summary
print(f"\n{'=' * 80}")
print("CROSS-VALIDATION SUMMARY")
print(f"{'=' * 80}\n")

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\n--- Training Metrics ---")
print(
    f"Mean Train Total Loss: {results_df['train_total_loss'].mean():.4f} ± {results_df['train_total_loss'].std():.4f}"
)
print(
    f"Mean Train Recon Loss: {results_df['train_recon_loss'].mean():.4f} ± {results_df['train_recon_loss'].std():.4f}"
)
print(
    f"Mean Train Clf Loss: {results_df['train_clf_loss'].mean():.4f} ± {results_df['train_clf_loss'].std():.4f}"
)

print("\n--- Validation Metrics ---")
print(
    f"Mean Val Total Loss: {results_df['val_total_loss'].mean():.4f} ± {results_df['val_total_loss'].std():.4f}"
)
print(
    f"Mean Val Recon Loss: {results_df['val_recon_loss'].mean():.4f} ± {results_df['val_recon_loss'].std():.4f}"
)
print(
    f"Mean Val Clf Loss: {results_df['val_clf_loss'].mean():.4f} ± {results_df['val_clf_loss'].std():.4f}"
)


Leave-One-Out: ID.4



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Found 2127 training spectrograms and 1111 validation spectrograms.
Dataset initialized with 2127 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 634, 'track211': 1493}
Dataset initialized with 1111 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 83, 'track211': 1028}


/Users/moritzfeik/Developer/ROAR/.pixi/envs/default/lib/python3.13/site-packages/lightning/pytorch/core/optimizer.py:259: Found unsupported keys in the lr scheduler dict: {'step'}. HINT: remove them from the output of `configure_optimizers`.

  | Name                     | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------------------
0 | classifier_criterion     | CrossEntropyLoss  | 0      | train | 0    
1 | reconstruction_criterion | MSELoss           | 0      | train | 0    
2 | encoder                  | Sequential        | 40.1 K | train | 0    
3 | global_pool              | AdaptiveAvgPool1d | 0      | train | 0    
4 | fc_latent                | Linear            | 16.5 K | train | 0    
5 | fc_decode                | Linear            | 16.5 K | train | 0    
6 | decoder                  | Sequential        | 29.6 K | train | 0    
7 | classifier               | Sequential        | 8.4 K  | train | 0    
-----------

Found 2127 training spectrograms and 1111 validation spectrograms.
Dataset initialized with 2127 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 634, 'track211': 1493}
Dataset initialized with 1111 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 83, 'track211': 1028}
                                                                           

...

Found 2849 training spectrograms and 389 validation spectrograms.
Dataset initialized with 2849 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 623, 'track211': 2226}
Dataset initialized with 389 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 94, 'track211': 295}
Epoch 41: 100%|██████████| 357/357 [00:07<00:00, 49.56it/s, v_num=0, val/total_loss=1.030, val/recon_loss=0.732, val/clf_loss=2.980, train/total_loss=0.475, train/recon_loss=0.474, train/clf_loss=0.0112] 

Results for Taycan:
  Train Total Loss: 0.4747
  Train Recon Loss: 0.4736
  Train Clf Loss: 0.0112
  Val Total Loss: 1.0308
  Val Recon Loss: 0.7324
  Val Clf Loss: 2.9840
  Train samples: 2849, Val samples: 389

Leave-One-Out: E-Golf



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Found 2610 training spectrograms and 628 validation spectrograms.
Dataset initialized with 2610 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 330, 'track211': 2280}
Dataset initialized with 628 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 387, 'track211': 241}



  | Name                     | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------------------
0 | classifier_criterion     | CrossEntropyLoss  | 0      | train | 0    
1 | reconstruction_criterion | MSELoss           | 0      | train | 0    
2 | encoder                  | Sequential        | 40.1 K | train | 0    
3 | global_pool              | AdaptiveAvgPool1d | 0      | train | 0    
4 | fc_latent                | Linear            | 16.5 K | train | 0    
5 | fc_decode                | Linear            | 16.5 K | train | 0    
6 | decoder                  | Sequential        | 29.6 K | train | 0    
7 | classifier               | Sequential        | 8.4 K  | train | 0    
-------------------------------------------------------------------------------
111 K     Trainable params
0         Non-trainable params
111 K     Total params
0.445     Total estimated model params size (MB)
29        Modules in train mode
0         M

Found 2610 training spectrograms and 628 validation spectrograms.
Dataset initialized with 2610 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 330, 'track211': 2280}
Dataset initialized with 628 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 387, 'track211': 241}
Epoch 29: 100%|██████████| 327/327 [00:06<00:00, 48.03it/s, v_num=0, val/total_loss=0.975, val/recon_loss=0.835, val/clf_loss=1.400, train/total_loss=0.501, train/recon_loss=0.501, train/clf_loss=0.00192]

Results for E-Golf:
  Train Total Loss: 0.5013
  Train Recon Loss: 0.5011
  Train Clf Loss: 0.0019
  Val Total Loss: 0.9749
  Val Recon Loss: 0.8349
  Val Clf Loss: 1.3997
  Train samples: 2610, Val samples: 628

CROSS-VALIDATION SUMMARY

left_out_vehicle  train_total_loss  train_recon_loss  train_clf_loss  val_total_loss  val_recon_loss  val_clf_loss  train_size  val_size
            ID.4          0.558235          0.